In [ ]:
"""
SAFE.zip -> training tiles pipeline (S1 VV/VH + S2 B02/B03/B04/B08), with filtering.

Key changes vs earlier attempts:
- NO full-scene S1 GeoTIFF export from SNAP (avoids 4GB GeoTIFF limit).
- Uses SNAP TC output in ENVI format: Sigma0_VV.img / Sigma0_VH.img inside *_tc.data/
- Warps S1 VV/VH and S2 SCL to the S2 10m grid, then tiles + filters + writes .npz.

Outputs:
  data/tiles_npz/<scene>_r<row>_c<col>.npz
Each NPZ contains: s1 (2,H,W), s2 (4,H,W), valid (H,W), meta (json str)

Assumes:
- You already have a working SNAP graph for S1 SAFE -> terrain-corrected DIM (GRAPH_TC)
  (this matches your “single chip works” notebook reference).
- SNAP gpt is installed and SNAP_GPT points to it.
"""

import os, json, zipfile, subprocess
from pathlib import Path

import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling

# -----------------------------
# User settings (EDIT THESE)
# -----------------------------
RAW_DIR   = Path("data/raw")
PROC_DIR  = Path("data/proc")
OUT_DIR   = Path("data/tiles_npz")

# Input SAFE.zip files
S1_ZIP = Path("/home/jakob/Bachelorprojekt/Kode/pipeline/data/raw/S1C_IW_GRDH_1SDV_20250819T053147_20250819T053212_003737_007770_1C38.SAFE.zip")
S2_ZIP = Path("/home/jakob/Bachelorprojekt/Kode/pipeline/data/raw/S2C_MSIL2A_20250819T104041_N0511_R008_T32VNH_20250819T155312.SAFE.zip")

# SNAP GPT + your existing graph that produces *_tc.dim + *_tc.data/Sigma0_*.img
# Example (adjust to your machine):
SNAP_GPT = Path.home() / "esa-snap" / "bin" / "gpt"
GRAPH_TC = Path("s1_grd_to_tc_dim.xml")  # SAFE -> TC .dim (your working one)

# Tiling (on S2 10m grid)
TILE   = 256          # 256 px at 10 m => 2.56 km
STRIDE = 256          # set smaller for overlap
MIN_VALID_FRAC = 1 # keep tile if at least this fraction is valid (clear + nonzero)

# Cloud/shadow/snow filtering using S2 SCL (Scene Classification Layer)
# Common SCL invalid classes: 3=cloud shadow, 8/9/10=clouds, 11=snow
SCL_INVALID = {3, 8, 9, 10, 11}

# Ocean/empty filtering using S1 texture (std of VV in dB over valid pixels)
OCEAN_STD_THR = 1.2   # tune after you run and inspect tile counts

# -----------------------------
# Housekeeping
# -----------------------------
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
def run(cmd):
    cmd = list(map(str, cmd))
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

def unzip_safe(zip_path: Path, out_dir: Path) -> Path:
    if not zip_path.exists():
        raise FileNotFoundError(f"Missing zip: {zip_path}")
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)
    safes = list(out_dir.glob("*.SAFE")) or list(out_dir.glob("**/*.SAFE"))
    if not safes:
        raise FileNotFoundError(f"No .SAFE found after unzip: {zip_path} -> {out_dir}")
    return safes[0]

def find_s2_paths(s2_safe: Path):
    # L2A tiles: bands are in GRANULE/.../IMG_DATA/R10m etc.
    r10_list = list(s2_safe.glob("**/IMG_DATA/R10m"))
    if not r10_list:
        raise FileNotFoundError("Could not find IMG_DATA/R10m in S2 SAFE")
    r10 = r10_list[0]

    b02 = list(r10.glob("*_B02_10m.jp2"))[0]
    b03 = list(r10.glob("*_B03_10m.jp2"))[0]
    b04 = list(r10.glob("*_B04_10m.jp2"))[0]
    b08 = list(r10.glob("*_B08_10m.jp2"))[0]

    r20 = list(s2_safe.glob("**/IMG_DATA/R20m"))[0]
    scl = list(r20.glob("*_SCL_20m.jp2"))[0]
    return b02, b03, b04, b08, scl

def read_s2_stack(b02,b03,b04,b08, window):
    arrs = []
    for p in [b02,b03,b04,b08]:
        with rasterio.open(p) as ds:
            arrs.append(ds.read(1, window=window))
    return np.stack(arrs, axis=0)  # (4,H,W)

def epsg_from_s2_b02(b02_10m: Path) -> str:
    import rasterio
    with rasterio.open(b02_10m) as ds:
        epsg = ds.crs.to_epsg()
        if epsg is None:
            raise ValueError(f"Could not determine EPSG from {b02_10m} (CRS={ds.crs})")
        return f"EPSG:{epsg}"

def warp_raster_to_ref(src_path: Path, ref_path: Path, out_path: Path, count: int, resampling):
    """Warp src raster (count bands) to match ref raster CRS/transform/shape."""
    with rasterio.open(ref_path) as ref, rasterio.open(src_path) as src:
        prof = ref.profile.copy()
        prof.update(
            driver="GTiff",
            count=count,
            dtype="float32",
            compress="deflate",
            predictor=2,
            tiled=True,
            blockxsize=256,
            blockysize=256,
        )
        dst = np.zeros((count, ref.height, ref.width), dtype=np.float32)

        if src.count < count:
            raise ValueError(f"{src_path} has {src.count} bands, expected >= {count}")

        for b in range(1, count+1):
            reproject(
                source=rasterio.band(src, b),
                destination=dst[b-1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref.transform,
                dst_crs=ref.crs,
                resampling=resampling,
            )

        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **prof) as out:
            out.write(dst)

def warp_s1_imgs_to_s2_grid(vv_path: Path, vh_path: Path, ref_10m: Path, out_path: Path):
    """Warp ENVI Sigma0_VV.img and Sigma0_VH.img onto S2 10m grid; write 2-band GeoTIFF."""
    with rasterio.open(ref_10m) as ref, rasterio.open(vv_path) as vv, rasterio.open(vh_path) as vh:
        prof = ref.profile.copy()
        prof.update(
            driver="GTiff",
            count=2,
            dtype="float32",
            compress="deflate",
            predictor=2,
            tiled=True,
            blockxsize=256,
            blockysize=256,
        )
        dst = np.zeros((2, ref.height, ref.width), dtype=np.float32)

        reproject(
            source=rasterio.band(vv, 1),
            destination=dst[0],
            src_transform=vv.transform, src_crs=vv.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.bilinear,
        )
        reproject(
            source=rasterio.band(vh, 1),
            destination=dst[1],
            src_transform=vh.transform, src_crs=vh.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.bilinear,
        )

        dst = 10.0 * np.log10(np.maximum(dst, 1e-10)).astype(np.float32)

        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **prof) as out:
            out.write(dst)

def scl_window_mask(scl_on_s2_10m: Path, window: Window) -> np.ndarray:
    """Return boolean valid mask for given window (True = keep)."""
    with rasterio.open(scl_on_s2_10m) as ds:
        scl = ds.read(1, window=window).astype(np.uint8)
    return ~np.isin(scl, list(SCL_INVALID))

# -----------------------------
# Pipeline steps
# -----------------------------
def process_s1_to_tc_imgs(s1_zip: Path, map_proj: str):
    """
    Unzip S1 SAFE.zip and run SNAP TC graph to produce:
      PROC_DIR/<name>_tc.dim
      PROC_DIR/<name>_tc.data/Sigma0_VV.img (+hdr)
      PROC_DIR/<name>_tc.data/Sigma0_VH.img (+hdr)
    Returns paths to vv_img and vh_img.
    """
    if not SNAP_GPT.exists():
        raise FileNotFoundError(f"SNAP GPT not found: {SNAP_GPT}")
    if not GRAPH_TC.exists():
        raise FileNotFoundError(f"GRAPH_TC not found: {GRAPH_TC}")

    s1_unzip_dir = RAW_DIR / s1_zip.stem.replace(".SAFE", "")
    s1_safe = unzip_safe(s1_zip, s1_unzip_dir)
    print("S1 SAFE:", s1_safe)

    base = s1_zip.stem.replace(".SAFE", "")
    s1_tc_dim = PROC_DIR / (base + "_tc.dim")

    if not s1_tc_dim.exists():
        run([SNAP_GPT, GRAPH_TC, f"-Pin={s1_safe}", f"-Pout={s1_tc_dim}", f"-PmapProjection={map_proj}"])
    else:
        print("S1 TC DIM exists:", s1_tc_dim)

    tc_data = PROC_DIR / (base + "_tc.data")
    vv = tc_data / "Sigma0_VV.img"
    vh = tc_data / "Sigma0_VH.img"

    if not vv.exists() or not vh.exists():
        raise FileNotFoundError(f"Missing VV/VH outputs in {tc_data}. Contents: {list(tc_data.glob('*'))[:10]}")

    return vv, vh

def prepare_s2(s2_zip: Path):
    """Unzip S2 SAFE.zip and return paths to 10m bands + 20m SCL."""
    s2_unzip_dir = RAW_DIR / s2_zip.stem.replace(".SAFE", "")
    s2_safe = unzip_safe(s2_zip, s2_unzip_dir)
    print("S2 SAFE:", s2_safe)
    b02,b03,b04,b08,scl = find_s2_paths(s2_safe)
    return s2_safe, b02,b03,b04,b08,scl

def warp_scl_to_ref_uint8(src_path: Path, ref_path: Path, out_path: Path):
    """Warp SCL (categorical) to ref grid and write as uint8."""
    with rasterio.open(ref_path) as ref, rasterio.open(src_path) as src:
        prof = ref.profile.copy()
        prof.update(
            driver="GTiff",
            count=1,
            dtype="uint8",
            compress="deflate",
            tiled=True,
            blockxsize=256,
            blockysize=256,
        )

        dst = np.zeros((ref.height, ref.width), dtype=np.uint8)

        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref.transform,
            dst_crs=ref.crs,
            resampling=Resampling.nearest,
        )

        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **prof) as out:
            out.write(dst, 1)


def align_to_s2_grid(vv_img: Path, vh_img: Path, b02_10m: Path, scl_20m: Path, tag: str):
    """
    Create:
      PROC_DIR/<tag>__S1_on_S2_10m.tif  (2-band float32)
      PROC_DIR/<tag>__SCL_on_S2_10m.tif (1-band uint8-ish in float container)
    """
    s1_on_s2 = PROC_DIR / f"{tag}__S1_on_S2_10m.tif"
    if not s1_on_s2.exists():
        print("Warping S1 VV/VH -> S2 grid:", s1_on_s2)
        warp_s1_imgs_to_s2_grid(vv_img, vh_img, b02_10m, s1_on_s2)
    else:
        print("S1 already aligned:", s1_on_s2)

    scl_on_s2 = PROC_DIR / f"{tag}__SCL_on_S2_10m.tif"
    if not scl_on_s2.exists():
        print("Warping SCL 20m -> 10m:", scl_on_s2)
        warp_scl_to_ref_uint8(scl_20m, b02_10m, scl_on_s2)
    else:
        print("SCL already aligned:", scl_on_s2)

    return s1_on_s2, scl_on_s2

def tile_and_write(s1_on_s2: Path, scl_on_s2: Path, b02,b03,b04,b08, scene_id: str):
    """Tile, filter, and write NPZ files."""
    with rasterio.open(b02) as ref, rasterio.open(s1_on_s2) as s1ds:
        H, W = ref.height, ref.width
        if (s1ds.height, s1ds.width) != (H, W):
            raise RuntimeError("S1 not aligned to S2 grid (shape mismatch).")

        wrote = 0
        skipped_valid = 0
        skipped_ocean = 0

        for r0 in range(0, H - TILE + 1, STRIDE):
            for c0 in range(0, W - TILE + 1, STRIDE):
                win = Window(c0, r0, TILE, TILE)

                # Read data
                s2 = read_s2_stack(b02,b03,b04,b08, win)                 # uint16
                s1 = s1ds.read([1,2], window=win).astype(np.float32)     # float32

                # Valid mask: clear sky + S2 nonzero + S1 not-nodata
                clear = scl_window_mask(scl_on_s2, win)
                nonzero = np.all(s2 > 0, axis=0)

                # S1 nodata check (tune threshold if needed)
                s1_ok = np.isfinite(s1[0]) & np.isfinite(s1[1]) & (s1[0] > -80) & (s1[1] > -80)
                valid = clear & nonzero & s1_ok

                # Optional: require strong S1 coverage in the tile
                s1_frac = float(s1_ok.mean())
                if s1_frac < 0.85:
                    skipped_valid += 1
                    continue


                valid_frac = float(valid.mean())
                if valid_frac < MIN_VALID_FRAC:
                    skipped_valid += 1
                    continue

                # Ocean/empty filter: low texture in VV dB (over valid pixels)
                vv_db = s1[0]
                score = float(np.nanstd(vv_db[valid]))
                if score < OCEAN_STD_THR:
                    skipped_ocean += 1
                    continue

                meta = {
                    "scene": scene_id,
                    "row0": int(r0), "col0": int(c0),
                    "tile": int(TILE), "stride": int(STRIDE),
                    "valid_frac": valid_frac,
                    "vv_db_std": score,
                    "crs": str(ref.crs),
                }

                out = OUT_DIR / f"{scene_id}_r{r0}_c{c0}.npz"
                np.savez_compressed(out,
                                    s1=s1,         # (2,H,W) VV,VH
                                    s2=s2,         # (4,H,W) B02,B03,B04,B08
                                    valid=valid.astype(np.uint8),
                                    meta=json.dumps(meta))
                wrote += 1

        print(f"\nScene: {scene_id}")
        print(f"Wrote tiles: {wrote}")
        print(f"Skipped (low valid frac): {skipped_valid}")
        print(f"Skipped (ocean/low texture): {skipped_ocean}")
        print(f"Output dir: {OUT_DIR.resolve()}")

def build_training_tiles(s1_zip: Path, s2_zip: Path):
    # 1) S2 first: unzip + find bands
    s2_safe, b02,b03,b04,b08,scl = prepare_s2(s2_zip)

    # 2) Auto CRS from S2
    map_proj = epsg_from_s2_b02(b02)
    print("Using mapProjection:", map_proj)

    # 3) S1: SAFE.zip -> TC (SNAP) -> ENVI outputs (pass mapProjection)
    vv_img, vh_img = process_s1_to_tc_imgs(s1_zip, map_proj)

    # 4) Align + tile
    scene_id = s2_zip.stem.replace(".SAFE", "")
    s1_on_s2, scl_on_s2 = align_to_s2_grid(vv_img, vh_img, b02, scl, tag=scene_id)
    tile_and_write(s1_on_s2, scl_on_s2, b02,b03,b04,b08, scene_id)

# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    build_training_tiles(S1_ZIP, S2_ZIP)


S1 SAFE: data/raw/S1C_IW_GRDH_1SDV_20250819T053147_20250819T053212_003737_007770_1C38/S1C_IW_GRDH_1SDV_20250819T053147_20250819T053212_003737_007770_1C38.SAFE
S1 TC DIM exists: data/proc/S1C_IW_GRDH_1SDV_20250819T053147_20250819T053212_003737_007770_1C38_tc.dim
S2 SAFE: data/raw/S2C_MSIL2A_20250819T104041_N0511_R008_T32VNH_20250819T155312/S2C_MSIL2A_20250819T104041_N0511_R008_T32VNH_20250819T155312.SAFE
Warping S1 VV/VH -> S2 grid: data/proc/S2C_MSIL2A_20250819T104041_N0511_R008_T32VNH_20250819T155312__S1_on_S2_10m.tif
Warping SCL 20m -> 10m: data/proc/S2C_MSIL2A_20250819T104041_N0511_R008_T32VNH_20250819T155312__SCL_on_S2_10m.tif

Scene: S2C_MSIL2A_20250819T104041_N0511_R008_T32VNH_20250819T155312
Wrote tiles: 781
Skipped (low valid frac): 983
Skipped (ocean/low texture): 0
Output dir: /home/jakob/Bachelorprojekt/Kode/pipeline/data/tiles_npz
